# YT-AI — generate stories

A 51.2M-parameter GPT trained from scratch on 2.44M synthetic short stories (852M GPT-2 tokens, 2 epochs, final val loss 2.15).

Run the three cells below top to bottom. Weights download automatically from [parthdabhi5195/yt-ai-slm](https://huggingface.co/parthdabhi5195/yt-ai-slm) on first use.

```
pip install torch tiktoken huggingface_hub
```

Training code and the full run (loss curve, sample output) are in `YT_AI.ipynb`.

## 1. YT-AI ARCHITECTURE

Copied from `YT_AI.ipynb` so the weights load exactly.

In [1]:
import torch # base PyTorch library
import torch.nn as nn # neural network module (ex. nn.Linear, nn.Embedding)
import torch.nn.functional as F # F.layer_norm, F.softmax, F.cross_entropy
import math # square root scaling
from dataclasses import dataclass # structured data containers
import numpy as np
from tqdm.auto import tqdm # progress bar generator
from contextlib import nullcontext # placeholder context manager
import os # filesystem operations

# Use LayerNorm instead of BatchNorm. LayerNorm is used in NLP when sentences come in variable length sequences.
# [5, 2, 7, 4, 1, 3]
# [8, 1, 6, X, X, X] why average over dimension 0 when columns near the end are mostly padding?
# [3, 9, X, X, X, X]
# [6, 4, 2, 7, 5, X]
class LayerNorm(nn.Module): # override nn.Module's __init__() and forward()
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim)) # gamma
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None # beta
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5) # gamma * (standarized feature row) + beta

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0 # ensures embedding dimension cleanly divides across all heads of attention
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias) # (768, 3 * 768). 768 inputs, and 3x to later split Q, K, V
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias) # (768, 768) linear layer that mixes head outputs back into residual stream
        # takes percentage of random elements in attention affinity matrix (QK / sqrt(dk)) and sets to 0 during training (not inference)
        self.attn_dropout = nn.Dropout(config.dropout)# prevents model from relying heavily on single token-to-token relationships. Forces tokens to learn features from other tokens. Prevents overfitting
        # takes output vector y (after linear projection) but before it gets added to residual stream, and randomly 0s channel dimensions
        self.resid_droupout = nn.Dropout(config.dropout) # encourages model to not heavily rely on entire attention output and preserve more information through residual path
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # prevents writing large (T, T) matrix into slow GPU VRAM. Rather, it creates smaller blocks inside fast SRAM
        self.flash = hasattr(F, 'scaled_dot_product_attention') # True or False flag to check if user's version supports FlashAttention (speeds attention while using less GPU memory)
        # create masked attention manually if not self.flash
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # x will be 3D
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2) # slices the 3 * C projection into individual Q, K, V tensors (B, T, C)

        # reshape each tensor to (B, n_head, T, dk) to align each attention head as independent matrix batch
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            # faster way to implement the else condition
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1))) # (B, n, T, dk) @ (B, n, dk, T) = (B, n, T, T)
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf')) # overwrites future tokens with -inf (necessary for softmax)
            att = F.softmax(att, dim=-1) # (B, n, T, T)
            att = self.attn_dropout(att) # (B, n, T, T)
            y = att @ v # (B, n, T, T) @ (B, n, T, dk) = (B, n, T, dk)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_droupout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias) # fully connected linear layer to 4x C dimension to facilitate deeper interactions
        self.gelu = nn.GELU() # Gaussian Error Linear Unit (nonlinearity like RELU or tanh)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias) # project from 4C into original C, with the assumption the model learned something new
        self.dropout = nn.Dropout(config.dropout) # prevent overfitting (model can't memorize training data)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x)))) # 4C projection on x, then perform nonlinearity, then bring back to original dimension, then perform dropout

class Block(nn.Module):
    # In accordance with nanoGPT architecture
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)

    # residual skip connection
    def forward(self, x):
         x = x + self.attn(self.ln1(x))
         x = x + self.mlp(self.ln2(x))
         return x

@dataclass # bundling all parameters into a single object
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

# High-level module that bundles together entire YT-AI transformer model
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
        wte = nn.Embedding(config.vocab_size, config.n_embd), # word token embedding (maps integer token to respective higher dimensional vector)
        wpe = nn.Embedding(config.block_size, config.n_embd), # word position embedding (so model knows sequence of words)
        drop= nn.Dropout(config.dropout),
        h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
        ln_f = LayerNorm(config.n_embd, config.bias) # final LayerNorm before projecting to logits
        ))

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False) # linear projection to generate raw next-token prediction logits
        # prevents model from wasting capacity learning symmetrical input/output mappings. transposes the matrices internally
        self.transformer.wte.weight = self.lm_head.weight # points output projection matrix to the same memory and weights as the word token embedding.

        self.apply(self._init_weights) # recursively iterate through each submodule and initialize the weights of each module
        for pn, p in self.named_parameters(): # for parameter name, parameter tensor in interator of all model parameters
            if (pn.endswith('c_proj.weight')): # targetting the weights at the end of a residual connection
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
                """
                Residual connections can compound variance if not checked. This can create activation saturation in GELU
                In a Gaussian Distribution, when you add by a constant, variance is unaffected.
                However, if you add by a variable, where each data point is added a variable number, variance can increase since extreme data points can compound.
                This is the case for our residual connections. So, we need to use scaling factors to bring variance back to 1, which is 1 / sqrt(2 * # of layers)
                """

    def _init_weights(self, module): # custom weight initialization to prevent vanishing gradients and exploding activations
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02) # start with a really low std to prevent saturation in GELU
            if module.bias is not None:
                nn.init.zeros_(module.bias) # so each next token predictions have equal probability at initialization
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean= 0.0, std= 0.02)

    # journey that data takes through the network (starts at text tokens, flows through each layer, and finally ends at next token prediction)
    def forward(self, idx, targets=None): # idx is input tensor containing token IDs (B, T)
        device = idx.device
        b, t  = idx.size() # (B, T)
        assert t <= self.config.block_size # if true, then continue. if false, raise error
        pos = torch.arange(0, t, dtype=torch.int64, device=device) # tensor [0, 1, 2, t-1] in CPU or GPU. so model sees "cat runs from dog" and "dog runs from cat" differently.
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos) # converts positions into vectors
        x = self.transformer.drop(tok_emb + pos_emb) # apply a drop to prevent overfitting

        for block in self.transformer.h:
            x = block(x) # loops through layernorm and attention mechanism

        x = self.transformer.ln_f(x) # apply final layer norm before making predictions

        if targets is not None: # training (targets) or inference (without targets)
            logits = self.lm_head(x) # (B, T, V), V = vocab_size
            # combines nn.LogSoftmax() and nn.NLLLoss()
            # when we pass in a matrix (B * T, V), are we going into every single row and calculating the -log(p), where p is the probability from correct next token (index in targets), and then averaging every -log(p) in B * T rows to get final loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1) # view logits as (B * T, V) and ground truths as (B * T, 1). In addition, skip any ground truths equal to -1
            return logits, loss
        else: # inference stage
            """
            grabs only the last token.
            then passes into lm_head which is a simple linear layer that takes in C for each B element.
            in production environments like Gemini, the model isn't just serving a single person's prompt at a time,
            it handles multiple independent user prompts at once. This is why we have the B dimension, so each
            prompt of T tokens can be processed at a single instance.
            but in our case, B dimension will just be 1 since we will be prompting one at a time.
            """
            logits = self.lm_head(x[:, [-1], :]) # takes (B, T, C) -> (B, 1, C), where 1 is the last token -> (B, 1, V)
            return logits, None # return None since we can't compute loss during inference


    @torch.no_grad() # disables gradient tracking during inference
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        idx (B, T),
        max_new_tokens (max token you want model to generate),
        temperature (sampling randomness),
        top_k (integer restricting sampling pool to top K most probable candidate tokens)

        Generating tokens given a conditional sequence of tokens
        """
        for _ in range(max_new_tokens):
            # as loop runs, idx grows longer. create a sliding window to keep context window with most recent generated token
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:] # negative slice to say "start from the end and get block_size back"
            logits, _ = self(idx_cond) # calls forward method on with current context window and returns (B, 1, V)
            # -1 mean drop dimension entirely, -1: means preserve it
            # Larger temp flattens out the ratio (making more random). Smaller temp sharpens distribution. after division, logits are exponentiated, which is why this happens (exponentiaion doesn't scale linearly)
            logits = logits[:, -1, :] / temperature # (B, V) / temperature.
            if top_k is not None: # top_k prevents model from predicting gibberish words
                # find top-k largest logit in each row. safety of min() if top-k is larger than vocab_size
                v, _ = torch.topk(logits, min(top_k, logits.size(-1))) # (B, K)
                logits[logits < v[:, -1:]] = float('-inf') # create a boolean mask for every logit in vocab_size that is less than lowest top_k
            probs = F.softmax(logits, dim=-1) #  softmax (exponentiate and normalize) over V, where probs is (B, V) and B is typically 1 for 1 prompt
            idx_next = torch.multinomial(probs, num_samples=1) # sample the next probable token
            idx = torch.cat((idx, idx_next), dim=1) # append to the input token sequence
        return idx


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the weights

In [2]:
!pip3 install tiktoken

In [3]:
# ---- Load the trained model ----
# Uses ./yt-ai-slm/best_model_params.pt if you have it, otherwise downloads the
# weights from Hugging Face (cached, so only the first run touches the network).
import os, torch, tiktoken

BEST = "yt-ai-slm/best_model_params.pt"
if not os.path.exists(BEST):
    from huggingface_hub import hf_hub_download
    BEST = hf_hub_download("parthdabhi5195/yt-ai-slm", "best_model_params.pt")

device = ("cuda" if torch.cuda.is_available()
          else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
          else "cpu") # apple silicon is plenty for inference

config = GPTConfig( # must match what was trained, or the weights will not load
    vocab_size = 50257,
    block_size = 512,
    n_layer = 8,
    n_head = 8,
    n_embd = 512,
    dropout = 0.0,
    bias = True
)
model = GPT(config).to(device)

enc = tiktoken.get_encoding("gpt2") # same tokenizer used for training
eot_id = enc.eot_token

print(f"ready -- device: {device}, weights: {BEST}")


ready -- device: mps, weights: /Users/parth/.cache/huggingface/hub/models--parthdabhi5195--yt-ai-slm/snapshots/9d4645fe161c5a0ef93d7b085325ba56b8ddfe83/best_model_params.pt


## 3. Generate

Re-run this cell as often as you like — the model stays loaded.

```python
write_stories(20)
write_stories(5, temperature=0.9)
write_stories(3, prompt='My sister called me at midnight.')
```

In [4]:
model.load_state_dict(torch.load(BEST, map_location=device)) # loads the checkpoint containing the best weights recorded onto gpu, cpu, or mps
model.eval() # swtich layers like dropout to evaluation mode during inference

@torch.no_grad() # disables gradient tracking during inference
def write_stories(n = 5, max_new_tokens=600, temperature=0.8, top_k=200, prompt=None):
    ids = [eot_id] + (enc.encode_ordinary(prompt) if prompt else []) # model was trained with <|endoftext|>, so keeping it at beginning signals model that we are generating a new story
    idx = torch.tensor([ids] * n, dtype=torch.int64, device=device) # creates n rows in B dimension, to generate independent stories
    done = torch.zeros(n, dtype=torch.bool, device=device) # creates a boolean tensor of flags indicating which of the n stories produced <|endoftext|> token

    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -config.block_size:]) # ensures the context passed into the model never exceeds maximum context window of 512. so keep only most recent tokens
        logits = logits[:, -1, :] / temperature # temp <1.0 sharpens tokens to most likely (more COHERENT STORIES), temp >1.0 flattens distribution (creating randomness)
        v, _ = torch.topk(logits, min(top_k, logits.size(-1))) # truncate sampling pool to top_k most probable words, setting the rest to -inf
        logits[logits < v[:, [-1]]] = float("-inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1) # sample next tokenID
        nxt = torch.where(done.unsqueeze(1), torch.full_like(nxt, eot_id), nxt) # id story stream already finished, force any further generated tokens to remain <|endoftext|>
        done |= nxt.squeeze(1) == eot_id # mark a story as finished when it samples eot_id
        idx = torch.cat((idx, nxt), dim=1) # append newly sampled tokenID to the right of existing sequence.
        if bool(done.all()): break # early exit as soon as every story in the batch_limit emits <|endoftext|>

    for row in idx.tolist():
        out = row[1:] # strip off the leading <|endoftext|> manually injected
        ended = eot_id in out
        if ended: out = out[:out.index(eot_id)] # detects whether model produced an end token. if so, truncate the token list at that exact index to remove trailining padding tokens
        text = enc.decode(out).strip() # convert the array of token IDs back into human readable text via GPT-2 TikToken tokenizer
        print(f"=== {len(text.split())} words, {'complete' if ended else 'TRUNCATED'} ===") # print if story naturally ended or was truncated
        print(text)
        print()




In [8]:
write_stories(100)

=== 259 words, complete ===
I knew then, watching their smug faces, that I was finally free. It started with a casual act of kindness, a shared cup of tea on the porch porch of our inherited sunroom. My sister, always so quiet, had gathered us for a celebratory bonfire, and as the fire station buzzed with activity, I noticed her phone, left carelessly on the table. A quick search revealed a dating app profile, one that detailed their clandestine communications, a stark contrast to the loving, shared family life they so carefully curated.

The air grew thick with unspoken accusations as our godmother, her face a mask of forced composure, tried to usher us away. My other sister, the one who’d always been so protective, pulled me aside, her voice a frantic whisper about how this was for the best, a necessary separation to protect us all from the fallout of our parents' increasingly erratic behavior.

Then came the bombshell, delivered with a quiet confession: my sister had been helping th